In [2]:
import sys
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

import json

from pydantic import BaseModel
from textwrap import dedent
from IPython.display import Math

from pathlib import Path

from utils.schema_json import ReportData

from pydantic import BaseModel
from time import perf_counter

Added to sys.path: C:\Users\lucat\VSCodeProjects\PRIN


In [3]:
load_dotenv()  # Load environment variables from .env file

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri

In [17]:
SYSTEM_PROMPT_FILE_NAME = "system_prompt_with_hints_1.txt"

TEST_DATA_FILE_NAME = "data_finetune_hints_openai_test.jsonl"
MODEL = 'ft:gpt-4.1-nano-2025-04-14:luca-tramonti::CXYq9uBA'
MODEL_NAME_SAVE = 'FT-gpt-4.1-nano-2025-04-14_TEMP_0.0'
TEMPERATURE = 0.0
PREDICTIONS_FILE_NAME = "predictions"

In [5]:
#Carica system prompt da file
system_prompt_path = Path('../models/prompts/' + SYSTEM_PROMPT_FILE_NAME)

with open(system_prompt_path, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

print(SYSTEM_PROMPT)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è estrarre le caratteristiche strutturate dal referto RM fornito e mappare i dati nello schema JSON sottostante.

Regole di output rigorose:

1. Restituisci esclusivamente un oggetto JSON valido. Nessun testo introduttivo, spiegazione, codice o commento fuori dal JSON.
2. Rispetta esattamente i tipi e i valori consentiti per ciascun campo (vedi elenco sotto).
  - Se un campo è numerico ma nel testo non compare un valore, scrivi null.
  - Se un campo ha opzioni predefinite ma non è chiaramente determinabile, scrivi null.
3. I campi booleani all’interno di sedi_linfonodi_locoregionali e sedi_linfonodi_non_locoregionali devono contenere solo true o false.
  - Usa true se la sede è esplicitamente citata come coinvolta/sospetta.
  - Usa false in tutti gli altri casi (anche se non menzionata).
4. Non aggiungere o inventare informazioni non presenti nel refert

# Load Test Data

In [6]:
# Load training data from file
test_data_path = Path('../data/ft-dataset/' + TEST_DATA_FILE_NAME)

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

# Generazione

## Preliminari generazione

In [7]:
class ReportText(BaseModel):
    report_text: str

In [8]:
if True:
    display(ReportText.model_json_schema())

    test_report = ReportText(report_text=test_data[0]['messages'][1]['content'])
    print(test_report.report_text)

{'properties': {'report_text': {'title': 'Report Text', 'type': 'string'}},
 'required': ['report_text'],
 'title': 'ReportText',
 'type': 'object'}

RM ADDOME INFERIORE (S/C MDC)


L'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .

IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.
ADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOADENOMEGALIA SEDE DI MALATTIA CON ASSE CORTO DI CIRCA 8 MM . ALTRI LINFONODI SOSPETTI SONO PRESENTI NEL MESORETTO ED UNO IN SEDE EXTRAMESORETATLE A SINISTRA ALLE ORE 4 ADESO ALLA FASCIA PERIRETTALE, AL SUO ESTERNO .
CISTI RADICOLARI AI LIVELLI S2 E S3.

CONCLUSIONI: STADIAZIONE LOCALE  NEOPLASIA DEL

In [9]:
# Funzione generatrice
def extract_data_from_report(report_text: str, system_prompt: str = SYSTEM_PROMPT, temperature: float = TEMPERATURE):
    response = client.responses.parse(
        model=MODEL,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=ReportData
    )
    return response

In [10]:
# Esempio
if False:
    result = extract_data_from_report(test_report.report_text)
    if False:
        print(result.output_text)
        print(result.output_parsed.model_dump(mode='json'))

## Generazione Risultati

In [11]:
testi_input = []
for r in test_data:
    for m in r['messages']:
        if m['role'] == 'user':
            testi_input.append(m['content'])

In [12]:
risultati = []
t1 = perf_counter()
for i, testo in enumerate(testi_input):
    print(f'...processando output {i}')
    output = extract_data_from_report(testo)
    if output is None:
        risultati.append('no output')
    else:
        risultati.append(output)
t2 = perf_counter()
print(f'Tempo totale: {int((t2-t1)//60)} minuti {int((t2-t1))%60} secondi')

...processando output 0
...processando output 1
...processando output 2
...processando output 3
...processando output 4
...processando output 5
...processando output 6
...processando output 7
...processando output 8
...processando output 9
...processando output 10
...processando output 11
...processando output 12
...processando output 13
...processando output 14
...processando output 15
...processando output 16
...processando output 17
...processando output 18
...processando output 19
...processando output 20
...processando output 21
...processando output 22
...processando output 23
...processando output 24
...processando output 25
...processando output 26
...processando output 27
...processando output 28
Tempo totale: 1 minuti 21 secondi


In [14]:
if True:
    print(risultati[0].output_text)
    display(risultati[0].output_parsed.model_dump(mode='json'))

{"morfologia":"solido_polipoide","posizione":"medio","spessore_parietale":null,"estensione_cranio_caudale":50,"distanza_oai":50,"riflessione_peritoneale_anteriore":null,"infiltrazione_tessuto_adiposo":"si_5mm_plus","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","coinvolgimento_riflessione_peritoneale":"no","coinvolgimento_fascia_mesorettale":"si","linfonodi_sospetti":0,"sedi_linfonodi_locoregionali":{"mesorettali":false,"rettali_superiori":false,"mesenterici_inferiori":false,"iliaci_interni":false,"otturatori":false,"sacrali":false,"inguinali_sotto_dentata":false},"sedi_linfonodi_non_locoregionali":{"inguinali":false,"iliaci_esterni":false,"iliaci_comuni":false,"paraortici":false,"altri":false},"depositi_tumorali":"si","numero_depositi":0,"emvi_esteso":"si","carcinosi_peritoneale":"si","lesioni_ossee":"no","stadio_T":"T4a","stadio_N":"N2a","stadio_N1c":false,"mrf":"+","emvi":"+","metastasi":"M1"}


{'morfologia': 'solido_polipoide',
 'posizione': 'medio',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 50,
 'distanza_oai': 50,
 'riflessione_peritoneale_anteriore': None,
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'si',
 'linfonodi_sospetti': 0,
 'sedi_linfonodi_locoregionali': {'mesorettali': False,
  'rettali_superiori': False,
  'mesenterici_inferiori': False,
  'iliaci_interni': False,
  'otturatori': False,
  'sacrali': False,
  'inguinali_sotto_dentata': False},
 'sedi_linfonodi_non_locoregionali': {'inguinali': False,
  'iliaci_esterni': False,
  'iliaci_comuni': False,
  'paraortici': False,
  'altri': False},
 'depositi_tumorali': 'si',
 'numero_depositi': 0,
 'emvi_esteso': 'si',
 'carcinosi_peritoneale': 'si',
 'lesioni_ossee': 'no',
 'stadio_T': 'T4a',
 'stadio_N': 'N2a',
 'stadio_N1c': False,
 'mrf

# Salva risultati

In [15]:
results_dicts = []
for r in risultati:
    results_dicts.append(
        {
            'model': MODEL,
            'temperature': TEMPERATURE,
            'min_p': 0,
            'prediction': r.output_text
        }
    )

In [18]:
with open(PREDICTIONS_FILE_NAME + '-' + MODEL_NAME_SAVE + '.jsonl', 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')